# 神经网络

In [32]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### 获取训练设备


In [33]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


### 定义神经网络类

In [34]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)  # 将其移动到device上
print(model)

X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Predicted class: tensor([7], device='cuda:0')


### 模型层数

In [35]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


### nn.压平

In [36]:
# 将每个 2D 28x28 图像转换为 784 个像素值的连续数组（保持小批量维度（dim=0））
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


### nn.线性

线性层 是一个模块，它使用存储的权重和偏置对输入应用线性变换

In [37]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


### nn.ReLU

非线性激活函数用于构建模型输入和输出之间复杂的映射关系。它们在进行线性变换之后应用，引入非线性，从而帮助神经网络学习各种各样的现象

In [38]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1356,  0.7632, -0.5552,  0.1033,  0.0354,  0.0503,  0.2919, -0.2318,
          0.8750,  0.3170,  0.0837,  0.5775, -0.3093,  0.1725,  0.3346,  0.3202,
         -0.1741,  0.0893,  0.1484,  0.5079],
        [-0.2201,  0.7889, -0.3541,  0.2287,  0.0039,  0.1841, -0.0351, -0.0733,
          0.8791,  0.3549, -0.2210,  0.1753, -0.0977, -0.3259,  0.0695,  0.5701,
          0.1405, -0.1187,  0.0774,  0.4597],
        [-0.1057,  0.4916, -0.2422,  0.1472,  0.2990, -0.0275,  0.1161, -0.4142,
          0.9798,  0.4569, -0.0391,  0.1474,  0.0058,  0.0484, -0.0586,  0.5707,
         -0.0970, -0.2684,  0.2411, -0.1152]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.7632, 0.0000, 0.1033, 0.0354, 0.0503, 0.2919, 0.0000, 0.8750,
         0.3170, 0.0837, 0.5775, 0.0000, 0.1725, 0.3346, 0.3202, 0.0000, 0.0893,
         0.1484, 0.5079],
        [0.0000, 0.7889, 0.0000, 0.2287, 0.0039, 0.1841, 0.0000, 0.0000, 0.8791,
         0.3549, 0.0000, 0.1753, 0.0000, 0.0000, 0.06

### nn.Sequential

nn.Sequential是一个有序的模块容器。数据会按照定义的顺序在所有模块中传递。可以使用顺序容器快速构建网络

In [39]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
print(logits)

tensor([[ 0.0315, -0.2204,  0.1715, -0.0280, -0.1489,  0.0109,  0.1711,  0.0014,
          0.1017, -0.4479],
        [-0.0807, -0.3648,  0.2749, -0.1362,  0.0073, -0.1699,  0.2329,  0.0945,
          0.2102, -0.6008],
        [-0.1163, -0.1808,  0.1757,  0.0320, -0.0157, -0.0193,  0.2077,  0.2034,
          0.1681, -0.5549]], grad_fn=<AddmmBackward0>)


### nn.Softmax

神经网络的最后一个线性层返回logits（取值范围为 [-∞, ∞] 的原始值），这些 logits 会被传递给 nn.Softmax模块。这些 logits 会被缩放到 [0, 1] 的值，表示模型对每个类别的预测概率。dim参数指定了值之和必须为 1 的维度

In [40]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[0.1053, 0.0819, 0.1211, 0.0992, 0.0879, 0.1032, 0.1211, 0.1022, 0.1130,
         0.0652],
        [0.0941, 0.0708, 0.1343, 0.0890, 0.1028, 0.0861, 0.1288, 0.1121, 0.1259,
         0.0560],
        [0.0879, 0.0824, 0.1177, 0.1019, 0.0972, 0.0968, 0.1215, 0.1210, 0.1168,
         0.0567]], grad_fn=<SoftmaxBackward0>)


### 模型参数

神经网络中的许多层都是参数化的，也就是说，它们都具有相关的权重和偏置，这些权重和偏置会在训练过程中进行优化。子类化会nn.Module自动跟踪模型对象中定义的所有字段，并使所有参数都可以通过模型的 `parameters()` 或 `named_parameters()` 方法访问

In [41]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0190, -0.0003,  0.0079,  ..., -0.0006,  0.0280, -0.0055],
        [-0.0053, -0.0160, -0.0263,  ..., -0.0338, -0.0285, -0.0157]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0222, -0.0224], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0099, -0.0101,  0.0076,  ..., -0.0285, -0.0354, -0.0002],
        [ 0.0304, -0.0198,  0.0325,  ...,  0.0216, -0.0062,  0.0424]],
       device='cuda:0', grad_fn=<Sl